
# 🧪 GPT JSONL Batch Runner (with retry / checkpoint / resume)

이 노트북은 `advanced_prompts_for_llm.jsonl`(행당 1개 프롬프트) 파일을 읽어 **동시 요청·재시도·체크포인트·재개**가 가능한 방식으로 GPT API를 호출합니다.

**필수 사전 준비**
- OpenAI Python SDK v1 설치: `pip install --upgrade openai`
- 환경변수 설정:  
  - PowerShell: `$env:OPENAI_API_KEY="sk-..."`  
  - macOS/Linux: `export OPENAI_API_KEY="sk-..."`

**입력 JSONL 포맷(라인별 객체)**
```json
{
  "product_name": "덴마크 하이그릭요거트 400g",
  "persona_key": 1,
  "system_prompt": "...",
  "user_prompt": "..."
}
```


In [2]:
# === Config ===
from pathlib import Path

INPUT_JSONL = Path("advanced_prompts_for_llm.jsonl")  # 업로드된 프롬프트 파일
OUTPUT_JSONL = Path("results_adv.jsonl")              # 결과 저장 위치

MODEL = "gpt-4o-mini"
CONCURRENCY = 6               # 동시 요청 개수(권장 5~8)
MAX_OUTPUT_TOKENS = 512
TEMPERATURE = 0.2
TOP_P = 1.0
TIMEOUT = 120                 # 요청 타임아웃(초)
RETRIES = 5
MAX_BACKOFF = 30.0            # 지수 백오프 상한(초)

print(INPUT_JSONL.exists(), INPUT_JSONL)

True advanced_prompts_for_llm.jsonl


In [3]:
# === Imports & helpers ===
import os, json, time, random, signal, asyncio
from typing import Dict, Any, List, Set
from dataclasses import dataclass

try:
    from openai import AsyncOpenAI
except Exception as e:
    raise RuntimeError("OpenAI SDK가 필요합니다. 설치: pip install --upgrade openai") from e

def to_jsonl_line(obj: Dict[str, Any]) -> str:
    return json.dumps(obj, ensure_ascii=False) + "\n"

def load_done_ids(jsonl_path: Path) -> Set[str]:
    done = set()
    if jsonl_path.exists():
        with jsonl_path.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    key = obj.get("id")
                    if key:
                        done.add(str(key))
                except Exception:
                    continue
    return done

def build_id(o: Dict[str, Any]) -> str:
    # product_name | persona_key 조합으로 유일 ID 생성
    return f"{o.get('product_name','').strip()}|{o.get('persona_key')}"

class GracefulKiller:
    def __init__(self):
        self.kill_now = False
        signal.signal(signal.SIGINT, self.exit_gracefully)
        try:
            signal.signal(signal.SIGTERM, self.exit_gracefully)
        except Exception:
            pass
    def exit_gracefully(self, *args):
        self.kill_now = True

In [4]:
# === Read JSONL ===
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                obj = json.loads(line)
            except Exception as e:
                raise ValueError(f"Invalid JSON on line {ln}: {e}")
            # 필수 키 확인
            for k in ("product_name", "persona_key", "system_prompt", "user_prompt"):
                if k not in obj:
                    raise ValueError(f"Missing key '{k}' on line {ln}")
            rows.append(obj)
    return rows

# 미리보기 (상위 3건)
preview = read_jsonl(INPUT_JSONL)[:3]
preview

[{'product_name': '덴마크 하이그릭요거트 400g',
  'persona_key': 1,
  'system_prompt': '당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다. 당신의 임무는 주어진 데이터를 종합적으로 분석하여, 가상 소비자의 구매 행동을 정밀하게 예측하고 그 결과를 JSON 객체로 생성하는 것입니다.',
  'user_prompt': '# INSTRUCTION (Advanced)\n먼저, 아래 \'사고 과정(Chain-of-Thought)\' 순서에 따라 당신의 분석을 단계별로 서술하세요.\n1. **페르소나 핵심 분석**: 주어진 페르소나의 `meta` 정보와 가중치가 가장 높은 속성인 **\'HMR 선호도\'** 를 바탕으로, 이 소비자의 가장 중요한 구매 동기가 무엇인지 정의합니다.\n2. **제품-페르소나 연결**: 위 분석을 바탕으로, \'제품 정보\'의 특징들이 이 페르소나에게 얼마나 매력적일지 긍정적/부정적 요인을 평가합니다.\n3. **시장 상황 적용**: \'시장 경쟁 환경\'의 가격 정보를 페르소나의 관점에서 어떻게 받아들일지 분석합니다.\n4. **종합 결론**: 위의 모든 분석을 종합하여, 구매 확률과 월별 구매 빈도에 대한 최종 예측을 내립니다.\n\n마지막으로, \'---\' 구분선 아래에 당신의 최종 예측을 # OUTPUT FORMAT에 맞는 JSON 형식으로만 생성하세요.\n\n# INPUT DATA\n## 1. 제품 정보\n{\n    "brand": "동원 F&B",\n    "product_name": "덴마크 하이그릭요거트 400g",\n    "category": "우유류 > 발효유 > 호상-중대용량",\n    "features": [\n        "건강식품",\n        "고단백",\n        "고소한맛",\n        "높은 만족도"\n    ],\n    "targeted_consumer": [\n        "유당불내증"\n    ],\n    "price

In [5]:
# === OpenAI call (JSON mode with fallback) ===
async def call_openai_json(
    client,
    model: str,
    system_prompt: str,
    user_prompt: str,
    max_output_tokens: int,
    temperature: float,
    top_p: float,
    timeout_s: int,
    use_json_mode: bool = True,
) -> Dict[str, Any]:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    kwargs = dict(
        model=model,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_output_tokens,
        timeout=timeout_s,
    )
    if use_json_mode:
        kwargs["response_format"] = {"type": "json_object"}

    resp = await client.chat.completions.create(**kwargs)
    choice = resp.choices[0]
    content = getattr(choice, "message", None).content if hasattr(choice, "message") else getattr(choice, "text", None)
    usage = getattr(resp, "usage", None)
    usage_dict = usage.model_dump() if hasattr(usage, "model_dump") else (usage.__dict__ if usage else None)
    return {
        "content": content,
        "model": resp.model,
        "id_resp": resp.id,
        "usage": usage_dict,
    }

In [6]:
# === Worker & main loop ===
async def worker(job: Dict[str, Any], sem: asyncio.Semaphore, client, args, out_fh):
    backoff = 1.0
    attempt = 0
    use_json_mode = True
    while True:
        if args["killer"].kill_now:
            return
        try:
            async with sem:
                result = await call_openai_json(
                    client=client,
                    model=args["model"],
                    system_prompt=job["system_prompt"],
                    user_prompt=job["user_prompt"],
                    max_output_tokens=args["max_output_tokens"],
                    temperature=args["temperature"],
                    top_p=args["top_p"],
                    timeout_s=args["timeout"],
                    use_json_mode=use_json_mode,
                )
            text = result["content"] or ""
            parsed = None
            parse_error = None
            try:
                parsed = json.loads(text)
            except Exception as e:
                parse_error = str(e)

            record = {
                "id": job["id"],
                "product_name": job["product_name"],
                "persona_key": job["persona_key"],
                "model": result.get("model"),
                "response_id": result.get("id_resp"),
                "usage": result.get("usage"),
                "output_raw": text,
                "output_json": parsed,
                "parse_error": parse_error,
                "ts": time.time(),
                "attempts": attempt + 1,
            }
            out_fh.write(to_jsonl_line(record))
            out_fh.flush()
            return
        except Exception as e:
            attempt += 1
            msg = str(e)
            # 모델/엔드포인트가 JSON 모드를 지원하지 않을 때 1회 폴백
            if "response_format" in msg or "json_object" in msg:
                use_json_mode = False
            if attempt > args["retries"]:
                record = {
                    "id": job["id"],
                    "product_name": job["product_name"],
                    "persona_key": job["persona_key"],
                    "error": msg,
                    "ts": time.time(),
                    "attempts": attempt,
                }
                out_fh.write(to_jsonl_line(record))
                out_fh.flush()
                return
            await asyncio.sleep(min(args["max_backoff"], backoff + random.uniform(0, 0.5)))
            backoff = min(backoff * 2.0, args["max_backoff"])

async def main_async():
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("환경변수 OPENAI_API_KEY 가 설정되어야 합니다.")

    rows = read_jsonl(INPUT_JSONL)
    for o in rows:
        o["id"] = build_id(o)

    done_ids = load_done_ids(OUTPUT_JSONL)
    pending = [r for r in rows if str(r["id"]) not in done_ids]
    total = len(rows)

    print(f"Total={total} | AlreadyDone={len(done_ids)} | Pending={len(pending)} | Concurrency={CONCURRENCY} | Model={MODEL}")
    if not pending:
        print("Nothing to do."); 
        return

    client = AsyncOpenAI()
    sem = asyncio.Semaphore(CONCURRENCY)
    killer = GracefulKiller()

    args = dict(
        model=MODEL,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        timeout=TIMEOUT,
        retries=RETRIES,
        max_backoff=MAX_BACKOFF,
        killer=killer,
    )

    OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)
    started = time.time()

    tasks = []
    with OUTPUT_JSONL.open("a", encoding="utf-8") as out_fh:
        for job in pending:
            tasks.append(asyncio.create_task(worker(job, sem, client, args, out_fh)))

        while True:
            await asyncio.sleep(2.0)
            done_n = sum(1 for t in tasks if t.done())
            rate = done_n / max(1e-9, (time.time() - started))
            print(f"[progress] {done_n}/{len(tasks)} | ~{rate:.2f} it/s")
            if done_n == len(tasks) or killer.kill_now:
                break

        await asyncio.gather(*tasks, return_exceptions=True)

    print("Finished. Output ->", OUTPUT_JSONL)

In [7]:
# === Run ===
import nest_asyncio, asyncio
nest_asyncio.apply()
await main_async()

Total=5445 | AlreadyDone=0 | Pending=5445 | Concurrency=6 | Model=gpt-4o-mini
[progress] 0/5445 | ~0.00 it/s
[progress] 0/5445 | ~0.00 it/s
[progress] 1/5445 | ~0.16 it/s
[progress] 6/5445 | ~0.74 it/s
[progress] 6/5445 | ~0.59 it/s
[progress] 8/5445 | ~0.66 it/s
[progress] 12/5445 | ~0.85 it/s
[progress] 12/5445 | ~0.74 it/s
[progress] 14/5445 | ~0.77 it/s
[progress] 14/5445 | ~0.69 it/s
[progress] 17/5445 | ~0.77 it/s
[progress] 18/5445 | ~0.74 it/s
[progress] 18/5445 | ~0.69 it/s
[progress] 19/5445 | ~0.67 it/s
[progress] 22/5445 | ~0.73 it/s
[progress] 24/5445 | ~0.74 it/s
[progress] 24/5445 | ~0.70 it/s
[progress] 25/5445 | ~0.69 it/s
[progress] 26/5445 | ~0.68 it/s
[progress] 29/5445 | ~0.72 it/s
[progress] 30/5445 | ~0.71 it/s
[progress] 31/5445 | ~0.70 it/s
[progress] 32/5445 | ~0.69 it/s
[progress] 34/5445 | ~0.70 it/s
[progress] 35/5445 | ~0.70 it/s
[progress] 36/5445 | ~0.69 it/s
[progress] 39/5445 | ~0.72 it/s
[progress] 40/5445 | ~0.71 it/s
[progress] 41/5445 | ~0.70 it/s



## 📦 결과 요약 CSV 만들기 (선택)
`results_adv.jsonl`에서 `id`와 `output_json`(또는 `output_raw`)를 뽑아 CSV로 요약본을 만듭니다.


In [8]:
# === Build snapshot CSV (optional) ===
import csv, json

SNAPSHOT_CSV = OUTPUT_JSONL.with_suffix(".snapshot.csv")

with OUTPUT_JSONL.open("r", encoding="utf-8") as rf, SNAPSHOT_CSV.open("w", newline="", encoding="utf-8") as wf:
    w = csv.writer(wf)
    w.writerow(["id", "product_name", "persona_key", "output_json", "parse_error"])
    for line in rf:
        try:
            obj = json.loads(line)
        except Exception:
            continue
        w.writerow([
            obj.get("id",""),
            obj.get("product_name",""),
            obj.get("persona_key",""),
            json.dumps(obj.get("output_json"), ensure_ascii=False) if obj.get("output_json") is not None else "",
            obj.get("parse_error","") or ""
        ])

SNAPSHOT_CSV

WindowsPath('results_adv.snapshot.csv')